# Hydrocarbon autoencoder workflow
This workflow uses the various codes in this directory to run the autoencoder simulations of hydrocarbons.

## Get database statistics
We start with the analysis of the contents of the `qm9_ch_only_full.xyz` database, which contains many thousands of small hydrocarbons from the QM9 database with up to 9 carbon atoms. The code analyzes the frequency of specific chemical formulas and prints a summary at the end

In [25]:
# These are just library imports
import numpy as np
from ase.io import read,write
from sklearn.neural_network import MLPRegressor
from sklearn.manifold import MDS
from sklearn.cluster import KMeans
from ase import Atoms
import nglview as nv

In [26]:
# Copyright (c) 2025 by Miguel A. Caro, Aalto University (miguel.caro@aalto.fi, mcaroba@gmail.com)

# This reads in a database of structures using ASE
db = read("qm9_ch_only_full.xyz", index=":") # https://doi.org/10.5281/zenodo.10925480


# This finds all the formulas in the database
formulas = {}
for atoms in db:
    formula = "C" + str(atoms.symbols.count("C")) + "H" + str(atoms.symbols.count("H"))
    if formula not in formulas:
        formulas[formula] = 1
    else:
        formulas[formula] += 1

# This figures out which formulas have the most isomers in the database and ranks the
# formulas according to number of isomers
ranked_formulas = []
for formula in sorted(formulas, key=formulas.get, reverse=True):
    ranked_formulas.append(formula)
#    print(formula, formulas[formula])

print("The most common formula is " + ranked_formulas[0] + " with " + str(formulas[ranked_formulas[0]]) + " entries")
print("The full list is " + str(list(zip(ranked_formulas, [formulas[ranked_formulas[i]] for i in range(0, len(formulas))]))))

The most common formula is C9H14 with 1234 entries
The full list is [('C9H14', 1234), ('C9H12', 1074), ('C9H16', 691), ('C9H10', 444), ('C8H12', 284), ('C8H14', 221), ('C9H18', 184), ('C8H10', 178), ('C9H8', 111), ('C8H16', 73), ('C7H12', 68), ('C7H10', 56), ('C8H8', 46), ('C9H20', 35), ('C7H14', 29), ('C7H8', 28), ('C6H10', 21), ('C8H18', 18), ('C6H12', 12), ('C8H6', 12), ('C6H8', 10), ('C7H16', 9), ('C6H6', 7), ('C5H8', 6), ('C5H10', 5), ('C6H14', 5), ('C9H6', 5), ('C9H4', 5), ('C5H12', 3), ('C7H4', 3), ('C7H6', 3), ('C4H10', 2), ('C4H6', 2), ('C4H8', 2), ('C5H4', 2), ('C1H4', 1), ('C2H2', 1), ('C2H6', 1), ('C3H4', 1), ('C3H8', 1), ('C3H6', 1), ('C4H2', 1), ('C5H6', 1), ('C6H2', 1), ('C8H2', 1)]


We are now going to take hydrocarbons with the most common formula and put them in a separate database (don't mind the ASE warning):

In [27]:
# This accumulates and writes out the structures of the isomers with the most entries in the database for their formula
new_db = []
for atoms in db:
    formula = "C" + str(atoms.symbols.count("C")) + "H" + str(atoms.symbols.count("H"))
    if formula == ranked_formulas[0]:
        new_db.append(atoms)

write("db_" + ranked_formulas[0] + ".xyz", new_db)

Now we are going to randomly pick one of the molecules in the database and take a look:

In [28]:
# Visualize a randomly picked molecule
nv.show_ase(new_db[np.random.choice(len(new_db))])

NGLWidget()

## Make more data entries

Because our database is relatively small, and neural networks are data-hungry,
we are going to "rattle" the existing structures to make more molecules, this
means we are going to introduce small displacements of the atoms (deviations
from the equilibrium positions that were stored in the database):

In [29]:
db_old = read("db_C9H14.xyz", index=":")

max_rattle = 0.05

db_new = []
for atoms in db_old:
    for i in range(0, 10):
        atoms_cp = atoms.copy()
        atoms_cp.positions += 2.*max_rattle*(np.random.sample([len(atoms), 3])-0.5)
        db_new.append(atoms_cp)

write("db_C9H14_rattled.xyz", db_new)

## Coulomb matrix descriptor

For ML applications, we need to represent our molecules mathematically. For this,
we will use the Coulomb matrix, which is a global descriptor:

$
M_{ij} = \begin{cases} \frac{1}{2} Z_i^{2.4} & \text{if} \quad i = j,
\\
\frac{Z_i Z_j}{r_{ij}} & \text{if} \quad i \neq j. \end{cases}
$

Because our neural network takes vector (1D) data, and the Coulomb matrix we have defined
is symmetric, we will store the matrix elements in the upper triangle. We define one function
to "flatten" the matrix array and another one to "unflatten" it, i.e., to return the vector
to matrix form. Note that we always have the same number of hydrogens and carbons and that
carbon and hydrogen information is not swapped in the array ordering:

In [30]:
####################################################################################
#
# User-editable fields
#
# Autoencoder structure; we define the encoder's architecture and the decoder uses
# a symmetric architecture which we do not define explicitly
# I.e., we set encoder_hidden_layers = [nn_1, nn_2, ..., nn_NE] (nn = number of neurons;
# NE = number of encoder hidden layers; i.e., nn_1 are the number of neurons in hidden layer 1)
# and the autoencoder architecture will be [nn_1, nn_2, ..., nn_NE, ..., nn_2, nn_1]
# It's important to note that nn_NE defines the number of dimensions in the encoding
# space, and it should be lower than the original number of dimensions of the feature
# vector. In this example, without any further modifications, our input vectors have
# dimensions 253, but this would depend on the number of atoms. When encoding, there
# is a balance you need to strike: high dimensions nn_NE will lead to less lossy
# compression (you retain more information), but will make the optimization in the
# embedding space more challenging. Lower nn_NE will make this optimization easier
# but will retain less information after compression.
encoder_hidden_layers = [400, 200, 50]
use_charge = False # Should we use the nuclear charges in the definition of the Coulomb matrix?
alpha = 0.001 # Strength of regularization term; default is 0.0001
####################################################################################


# Don't change things below unless you know what you're doing


# This reads in a database of structures using ASE
#db = read("qm9_ch_only_full.xyz", index=":")
#db = read("db_C9H14.xyz", index=":")
db = read("db_C9H14_rattled.xyz", index=":")
#db = read("db_C9H14_train.xyz", index=":")


# This function computes the Coulomb matrix for an ASE's Atoms object
def coulomb_matrix(atoms, use_charge, sort_by_species=True):
    cm = np.zeros([len(atoms), len(atoms)])
    dist = np.zeros([len(atoms), len(atoms)])
    if use_charge:
        Z = atoms.numbers
    else:
        Z = np.zeros(len(atoms)) + 1.
    for i in range(0, len(atoms)):
        cm[i,i] = 0.5 * Z[i]**2.4
        dist[i,i] = 0.
        for j in range(i+1, len(atoms)):
            dist[i, j] = atoms.get_distance(i,j)
            dist[j, i] = dist[i, j]
            cm[i, j] = Z[i]*Z[j] / dist[i, j]
            cm[j, i] = cm[i, j]
    if sort_by_species:
        cm_sort_rows = cm[np.argsort(atoms.numbers)[::-1], :]
        cm_sorted = cm_sort_rows[:, np.argsort(atoms.numbers)[::-1]]
        return cm_sorted, dist
    else:
        return cm, dist


# This function retains the upper triangle (without the diagonal elements) of the
# Coulmb matrix and expresses it as a vector
def flatten_upper_triangle(m):
    flat = []
    assert m.shape[0] == m.shape[1]
    for i in range(0, m.shape[0]):
        for j in range(i+1, m.shape[1]):
            flat.append(m[i,j])
    return np.array(flat)

def unflatten_upper_triangle(flat, use_charge, nC, nH):
    N = int(1+np.sqrt(1+8*len(flat)))//2
    m = np.zeros([N,N])
    dist = np.zeros([N,N])
    k = 0
    for i in range(0, N):
        if i < nC:
            if use_charge:
                Zi = 6.
            else:
                Zi = 1.
        else:
            Zi = 1.
        m[i,i] = 0.5*Zi**2.4
        for j in range(i+1, N):
            if j < nC:
                if use_charge:
                    Zj = 6.
                else:
                    Zj = 1.
            else:
                Zj = 1.
            dist[i, j] = np.max([0., Zi*Zj/flat[k]])
            dist[j, i] = np.max([0., Zi*Zj/flat[k]])
            m[i,j] = flat[k]
            m[j,i] = flat[k]
            k += 1
    return m, dist

Let's visualize one of the structures in the regular representation (here I pick the first structure in the database)

In [31]:
nv.show_ase(db[0])

NGLWidget()

This is a sanity check to ensure we can embed the structure back into Cartesian space

In [32]:
# Now we make Coulomb, then flatten, then unflatten, then embed, then plot
cm, dist = coulomb_matrix(db[0], use_charge)
v = flatten_upper_triangle(cm)
cm, dist = unflatten_upper_triangle(v, use_charge, 9, 14)
mds = MDS(n_components=3, dissimilarity="precomputed", n_init=100)
xyz = mds.fit_transform(dist)
at = Atoms("C9H14", positions = xyz)
nv.show_ase(at)

NGLWidget()

## Artificial neural network

Here we train our neural network. Note that we defined the architecture in above when we declared `encoder_hidden_layers`. This steps takes around a minute with the default settings.

In [33]:
# This is the Relu activation function that takes a scalar as argument
def relu(x):
    return np.max([0., x])


# This builds the descriptors for all the isomers of the chosen formula
desc = []
for atoms in db:
    cm, dist = coulomb_matrix(atoms, use_charge)
    desc.append(flatten_upper_triangle(cm))


# This builds the neural network object and trains it according to an autoencoder architecture
hidden_layers = []
for i in range(0, len(encoder_hidden_layers)):
    hidden_layers.append(encoder_hidden_layers[i])

for i in range(len(encoder_hidden_layers)-2, -1, -1):
    hidden_layers.append(encoder_hidden_layers[i])

print("Your chosen autoencoder architecture is " + str(hidden_layers))

print("Training neural network...")
ae = MLPRegressor(solver='adam', hidden_layer_sizes=hidden_layers, max_iter=2000, random_state=1, alpha=alpha)
ae.fit(desc, desc)

weights = ae.coefs_
bias = ae.intercepts_
print("   ... done.")

Your chosen autoencoder architecture is [400, 200, 50, 200, 400]
Training neural network...
   ... done.


With the weights we obtained from the fit, we are going to define three functions,
one to "autoencoder", i.e., to compress and uncrompress, as well as an encoder
function to project the descriptor onto the latent space and a decoder function
to bring the latent space representation back to the input feature space (i.e.,
the Coulomb matrix representation).

In [34]:
# Define the function to autoencode
def autoencode(weights, bias, this_desc):
    v = this_desc.copy()
    for i in range(0, len(bias)):
        v_new = np.dot(np.transpose(weights[i]), v) + bias[i]
        if i < len(bias)-1:
            v = np.array([relu(a) for a in v_new])
        else:
            v = v_new
    return v


# Define the function to encode (assumes the NN has symmetric layer architecture)
def encode(weights, bias, this_desc):
    v = this_desc.copy()
    for i in range(0, len(bias)//2):
        v_new = np.dot(np.transpose(weights[i]), v) + bias[i]
        if i < len(bias)//2-1:
            v = np.array([relu(a) for a in v_new])
        else:
            v = v_new
    return v


# Define the function to decode (assumes the NN has symmetric layer architecture)
def decode(weights, bias, this_compressed_desc):
    v_new = this_compressed_desc.copy()
    v = np.array([relu(a) for a in v_new])
    for i in range(len(bias)//2, len(bias)):
        v_new = np.dot(np.transpose(weights[i]), v) + bias[i]
        if i < len(bias)-1:
            v = np.array([relu(a) for a in v_new])
        else:
            v = v_new
    return v

## Testing the model
First we calculate the RMSE within the *training* database. This is not too bad because
we're just looking at how well we can compress, rather than how well we can predict other
properties, or how well our model generalizes (i.e., outside the training data).

In [35]:
RMSE_train = 0.
for i in range(0, len(desc)):
    v = autoencode(weights, bias, desc[i])
    RMSE_train += np.dot(desc[i]-v, desc[i]-v)

RMSE_train = np.sqrt(RMSE_train/len(desc))
print("RMSE_train = " + str(RMSE_train))

RMSE_train = 0.689682060800877


Now let's do this on a test database. With default settings, the rattled database
was the training one so the original (non-rattled) database can be used for testing.
We will store all the vector descriptors in variable `desc` for convenience.

In [36]:
db_test = read("db_C9H14.xyz", index=":")
#db_test = read("db_C9H14_test.xyz", index=":")
desc = []
for atoms in db_test:
    cm, dist = coulomb_matrix(atoms, use_charge)
    desc.append(flatten_upper_triangle(cm))

RMSE_test = 0.
for i in range(0, len(desc)):
    v = autoencode(weights, bias, desc[i])
    RMSE_test += np.dot(desc[i]-v, desc[i]-v)

RMSE_test = np.sqrt(RMSE_test/len(desc))
print("RMSE_test = " + str(RMSE_test))

RMSE_test = 0.6733332936662947


Again for convenience, we will store the compressed descriptors:

In [37]:
# Create a library of compressed descriptors
compressed_lib = []
for i in range(0, len(desc)):
    v = encode(weights, bias, desc[i])
    compressed_lib.append(v)

We now do some visual checks of the quality of the embedding by encoding and
decoding the descriptors (use the `autoencode` function). We rancomly pick a
molecule:

In [38]:
i_rand = np.random.choice(len(desc))
m0, dist0 = unflatten_upper_triangle(desc[i_rand], use_charge, 9, 14) # Coulomb matrix of first structure
v = autoencode(weights, bias, desc[i_rand])
m, dist = unflatten_upper_triangle(v, use_charge, 9, 14) # Autoencoded Coulomb matrix of first structure

To draw the structures with the ball and stick representation, we need the Cartesian coordinates,
but we only have the distances (from the Coumb matrix). These distances are enough information to
retrieve the Cartesian coordinates and we use a dimensionality reduction method, multidimensional
scaling (MDS), to do the transformation from distances to Cartesian coordinates.

$
L (\{x_i\}, \{y_i\}, \{z_i\}) = \sum\limits_{i=1}^{N_\text{points}} \sum\limits_{j=i+1}^{N_\text{points}} \left( d_{0,ij} - d_{ij}  \right)^2,
$
with
$
\text{with } d_{ij} = \sqrt{(x_i - x_j)^2 + (y_i - y_j)^2 + (z_i - z_j)^2}.
$

MDS estimates the optimal $\{(x,y,z)_i\}$ by minimizing the difference between the
new distances $\{d_{ij}\}$ and the old distances $\{d_{0,ij}\}$, i.e., by minimizing the
*objective function* $L$.

In [40]:
# Get Cartesian coordinates from Coulomb matrix
mds = MDS(n_components=3, dissimilarity="precomputed", n_init=100)
xyz0 = mds.fit_transform(dist0) # Original CM
xyz = mds.fit_transform(dist) # Autoencoded CM

This is what the original molecule looks like:

In [41]:
atoms0 = Atoms("C9H14", positions=xyz0)
nv.show_ase(atoms0)

NGLWidget()

And this is what our "autoencoded" molecule looks like:

In [42]:
atoms = Atoms("C9H14", positions=xyz)
nv.show_ase(atoms)

NGLWidget()

## Data clustering

We are going to cluster the data so that we can visualize the "characteristic" molecules.
This is done by finding the coordinate of each cluster's centroid in the latent space and
then representing that latent space coordinate in our familiar Cartesian space. We will
use the k-means algorith for the clustering (and pick the number of clusters that we want):

In [43]:
# Let's cluster the data in the latent space with k-means
kmeans = KMeans(n_clusters=5, random_state=0, n_init=100).fit(compressed_lib)
labels = kmeans.labels_ # List with the cluster number each molecule belongs to
centers = kmeans.cluster_centers_ # The coordinates in latent space of the centroids

# Turn these centers into molecules (the most representative)
at_centers = []
for i in range(0, len(centers)):
    v = decode(weights, bias, centers[i])
    m, dist = unflatten_upper_triangle(v, use_charge, 9, 14)
    xyz = mds.fit_transform(dist)
    atoms = Atoms("C9H14", positions=xyz)
    at_centers.append(atoms)

This is the first one:

In [44]:
nv.show_ase(at_centers[0])

NGLWidget()

The second one:

In [45]:
nv.show_ase(at_centers[1])

NGLWidget()

You can check the others by replacing the centroid index in the code above.